In [ ]:
from datasets import load_dataset

# there are all "positive" pairs"
dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")


train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5424 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5445 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5516 [00:00<?, ? examples/s]

In [ ]:
len(dataset['train'])

5424

In [ ]:
dataset['train'][0]

{'mention': 'Naloxone',
 'mention_text': 'Naloxone reverses the antihypertensive effect of clonidine.',
 'entity': 'Naloxone',
 'aliases': 'Abello Brand of Naloxone Hydrochloride Boots Bristol Myers Squibb Bristol-Myers Curamed Naloxon Dihydride Endo Hydrobromide Lamepro MRZ 2593 Br 2593-Br 2593Br MRZ-2593 MRZ2593 Nalone ratiopharm Naloxon-ratiopharm (5 beta,9 alpha,13 alpha,14 alpha)-Isomer Naloxonratiopharm Narcan Narcanti SERB United Drug',
 'definition': 'A specific opiate antagonist that has no agonist activity. It is a competitive antagonist at mu, delta, and kappa opioid receptors.\n    ',
 'id': 'MESH:D009270'}

# Constructing negative pairs

In [ ]:
# I want to have 4 times as many negative pairs as positive pairs
# There are several negative sampling techinques
# for now take one mention name from the positives and one entity name from the positives that don't match
# make it sample randomly from the positives
import random

def create_negative_pairs(positive_pairs):
  negative_pairs = []
  while len(negative_pairs) < 4 * len(positive_pairs):
      entry = random.choice(positive_pairs)
      mention, mention_text, mention_id = entry['mention'], entry['mention_text'], entry['id']

      entry = random.choice(positive_pairs)
      entity, definition, entity_id = entry['entity'], entry['definition'], entry['id']

      if mention_id != entity_id:
          entry = {
              'mention': mention,
              'mention_text': mention_text,
              'entity': entity,
              'definition': definition,
              'id': mention_id
          }
          negative_pairs.append(entry)

  return negative_pairs


In [ ]:
negative_pairs_test = create_negative_pairs(dataset["test"])
negative_pairs_train = create_negative_pairs(dataset["train"])
negative_pairs_validation = create_negative_pairs(dataset["validation"])

# Combining Positive and Negative pairs, and adding labels


In [ ]:
def add_labels(positive_pairs, negative_pairs):
  training_data = []
  for entry in positive_pairs:
      entry["label"] = 1
      training_data.append(entry)

  for entry in negative_pairs:
      entry["label"] = 0
      training_data.append(entry)

  return training_data

In [ ]:
train_data = add_labels(dataset['train'], negative_pairs_train)
test_data = add_labels(dataset['test'], negative_pairs_test)
validation_data = add_labels(dataset['validation'], negative_pairs_validation)

In [ ]:
# load the new datasets
from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)
validation_dataset = Dataset.from_list(validation_data)

dataset_dict = DatasetDict({
    'train': train_dataset,
    'test': test_dataset,
    'validation': validation_dataset,
})

In [ ]:
# sanity check entries from splits
# first come all the positive pairs, after that the negatives
print(dataset_dict["train"][0])
print(dataset_dict["test"][0])
print(dataset_dict["validation"][-1])


{'mention': 'Naloxone', 'mention_text': 'Naloxone reverses the antihypertensive effect of clonidine.', 'entity': 'Naloxone', 'aliases': 'Abello Brand of Naloxone Hydrochloride Boots Bristol Myers Squibb Bristol-Myers Curamed Naloxon Dihydride Endo Hydrobromide Lamepro MRZ 2593 Br 2593-Br 2593Br MRZ-2593 MRZ2593 Nalone ratiopharm Naloxon-ratiopharm (5 beta,9 alpha,13 alpha,14 alpha)-Isomer Naloxonratiopharm Narcan Narcanti SERB United Drug', 'definition': 'A specific opiate antagonist that has no agonist activity. It is a competitive antagonist at mu, delta, and kappa opioid receptors.\n    ', 'id': 'MESH:D009270', 'label': 1}
{'mention': 'Famotidine', 'mention_text': 'Famotidine-associated delirium. A series of six cases.', 'entity': 'Famotidine', 'aliases': 'Famotidine Hydrochloride MK 208 MK-208 MK208 Pepcid YM 11170 YM-11170 YM11170', 'definition': 'A competitive histamine H2-receptor antagonist. Its main pharmacodynamic effect is the inhibition of gastric secretion.\n    ', 'id': '

In [ ]:
pairs = dataset_dict['train']
val_pairs = dataset_dict['validation']

# Training

In [ ]:
from sentence_transformers import SentenceTransformer, models, InputExample, losses
from torch.utils.data import DataLoader

model_name = 'cambridgeltl/SapBERT-from-PubMedBERT-fulltext'
word_embedding_model = models.Transformer(model_name)
pooling_model = models.Pooling(
    word_embedding_model.get_word_embedding_dimension(),
    pooling_mode='cls'
)
model = SentenceTransformer(modules=[word_embedding_model, pooling_model])



config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: cambridgeltl/SapBERT-from-PubMedBERT-fulltext
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [ ]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'

model.to(device)

Using device: cuda


SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

### Training can take a very long time so let's check what are the bottlenecks:

In [ ]:
import time
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, models

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

texts = ["This is a short test sentence."] * 256   # increase to mimic load

# tokenization timing
t0 = time.perf_counter()
_ = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
t1 = time.perf_counter()
print(f"Tokenize {len(texts)} samples: {t1-t0:.3f}s ({len(texts)/(t1-t0):.1f} samples/s)")

# model encode timing (warmup + measured)
_ = model.encode(texts[:8], convert_to_tensor=True)  # warmup
torch.cuda.synchronize() if torch.cuda.is_available() else None
t0 = time.perf_counter()
_ = model.encode(texts[:8], convert_to_tensor=True)
torch.cuda.synchronize() if torch.cuda.is_available() else None
t1 = time.perf_counter()
print(f"Encode 8 samples: {t1-t0:.3f}s ({8/(t1-t0):.1f} samples/s)")

### We can see that tokenisation is not the bottleneck (However, it is still very slow!)

## Preparing data for training

In [ ]:
# 2. Prepare Data (List of InputExamples)

# util function to get context window around the mention in the text
!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/text_preprocessing.py
from text_preprocessing import get_mt_window


train_examples = []
for m, mt, e, d, l in zip(
    pairs["mention"],
    pairs["mention_text"],
    pairs["entity"],
    pairs["definition"],
    pairs["label"]):
    # We pass TWO strings. SentenceTransformer handles the [SEP] and tokenization.
    # Text 1: Mention + Context | Text 2: Entity + Definition

    # pre-process the mention context and entity definition because they can be very long
    # mention context: find the mention in the context, then keep only the window 64 chars to the left and 64 chars to the right (the right will include the mention)
    mt_window = get_mt_window(m, mt)

    # definition: just take first 128 chars
    d = d[:128]

    train_examples.append(InputExample(
        texts=[f"{m} [SEP] {mt_window}", f"{e} [SEP] {d}"], # take only the first 128 chars  of mention text and definition
        label=float(l)
    ))

In [ ]:
# 3. Training logic
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16, num_workers=4, pin_memory=True)
train_loss = losses.ContrastiveLoss(model=model)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# 4. Fit
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    use_amp=True,
    show_progress_bar=True # This enables the tqdm bar
)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.010457
1000,0.006635


Step,Training Loss
500,0.010457
1000,0.006635
1500,0.005398
2000,0.004296
2500,0.003550
3000,0.003444
3500,0.003151
4000,0.002357
4500,0.002499
5000,0.002379


In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Login into Hugging Face Hub
hf_token = userdata.get('HF_TOKEN') # If you are running inside a Google Colab
login(hf_token)


# create a new huggingFace repo for this model
model.push_to_hub("context_fine-tuned-SapBERT2")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...18fbncx/model.safetensors:   0%|          | 17.4kB /  438MB            

'https://huggingface.co/Stevenf232/context_fine-tuned-SapBERT1/commit/e3b01cf80509d9e5fb24d1e7e24605973fc76e5e'